In [3]:
"""
LDA (Latent Dirichlet Allocation) 토픽 모델링 구현
Opinosis 데이터셋을 활용한 코드
"""

import os, glob, re
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from pycaret.clustering import * # setup, create_model, assign_model

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
import warnings

warnings.filterwarnings("ignore")

from utils import preprocessing

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\TJ\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [4]:
def LDATopicModeling(n_topics=6, random_state=23):
    """
    Parameters:
    -----------
    n_topics : int
        추출할 토픽의 개수 (K)
    random_state : int
        재현성을 위한 랜덤 시드
    """
    n_topics = n_topics
    random_state = random_state
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words("english"))

    return lemmatizer, stop_words


# ==========================================
# 방법 2: Scikit-learn을 이용한 LDA
# ==========================================
def train_sklearn_lda(documents, n_topics=6, random_state=23):
    """
    Scikit-learn을 이용한 LDA 모델 학습

    Parameters:
    -----------
    documents : list of str
        원본 문서 리스트

    Returns:
    --------
    lda_model : sklearn.decomposition.LatentDirichletAllocation
        학습된 LDA 모델
    vectorizer : sklearn.feature_extraction.text.CountVectorizer
        벡터라이저
    dtm : sparse matrix
        문서-단어 행렬
    """
    # CountVectorizer로 DTM 생성
    vectorizer = CountVectorizer(
        max_df=0.8,
        min_df=2,
        stop_words="english",
        lowercase=True,
        token_pattern="[a-zA-Z\-][a-zA-Z\-]{2,}",
    )

    dtm = vectorizer.fit_transform(documents)

    # LDA 모델 학습
    lda_model = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=random_state,
        max_iter=50,
        learning_method="online",
        n_jobs=-1,
    )

    lda_model.fit(dtm)

    # Perplexity 출력
    perplexity = lda_model.perplexity(dtm)
    print(f"Scikit-learn LDA Perplexity: {perplexity:.4f}")

    return lda_model, vectorizer, dtm


def get_document_topics_sklearn(lda_model, dtm):
    """
    각 문서의 토픽 분포 추출 (Scikit-learn)

    Returns:
    --------
    doc_topic_dist : numpy.ndarray
        문서-토픽 분포 행렬 (n_documents x n_topics)
    """
    doc_topic_dist = lda_model.transform(dtm)
    return doc_topic_dist


# ==========================================
# 결과 시각화 및 분석
# ==========================================

def display_topics(model, feature_names=None, n_top_words=10, model_type="sklearn"):
    """
    각 토픽의 주요 단어 출력

    Parameters:
    -----------
    model : LDA model
        학습된 LDA 모델
    feature_names : list
        단어 리스트 (sklearn의 경우)
    n_top_words : int
        출력할 단어 개수
    model_type : str
        'gensim' 또는 'sklearn'
    """
    print(f"\n{'='*60}")
    print(f"주요 토픽 및 키워드 ({model_type.upper()})")
    print(f"{'='*60}\n")

    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        print(f"토픽 {topic_idx + 1}:")
        print(f"  {', '.join(top_words)}\n")

In [5]:
def train_sklearn_lda(documents, n_topics=5, random_state=42):
    """
    Scikit-learn을 이용한 LDA 모델 학습

    Parameters:
    -----------
    documents : list of str
        원본 문서 리스트

    Returns:
    --------
    lda_model : sklearn.decomposition.LatentDirichletAllocation
        학습된 LDA 모델
    vectorizer : sklearn.feature_extraction.text.CountVectorizer
        벡터라이저
    dtm : sparse matrix
        문서-단어 행렬
    """
    # CountVectorizer로 DTM 생성
    vectorizer = CountVectorizer(
        max_df=0.8,
        min_df=2,
        stop_words="english",
        lowercase=True,
        token_pattern="[a-zA-Z\-][a-zA-Z\-]{2,}",
    )

    dtm = vectorizer.fit_transform(documents)

    # LDA 모델 학습
    lda_model = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=random_state,
        max_iter=50,
        learning_method="online",
        n_jobs=-1,
    )

    lda_model.fit(dtm)

    # Perplexity 출력
    perplexity = lda_model.perplexity(dtm)
    print(f"Scikit-learn LDA Perplexity: {perplexity:.4f}")

    return lda_model, vectorizer, dtm


def get_document_topics_sklearn(lda_model, dtm):
    """
    각 문서의 토픽 분포 추출 (Scikit-learn)

    Returns:
    --------
    doc_topic_dist : numpy.ndarray
        문서-토픽 분포 행렬 (n_documents x n_topics)
    """
    doc_topic_dist = lda_model.transform(dtm)
    return doc_topic_dist


# ==========================================
# 결과 시각화 및 분석
# ==========================================


def display_topics(model, feature_names=None, n_top_words=10, model_type="sklearn"):
    """
    각 토픽의 주요 단어 출력

    Parameters:
    -----------
    model : LDA model
        학습된 LDA 모델
    feature_names : list
        단어 리스트 (sklearn의 경우)
    n_top_words : int
        출력할 단어 개수
    model_type : str
        'gensim' 또는 'sklearn'
    """
    print(f"\n{'='*60}")
    print(f"주요 토픽 및 키워드 ({model_type.upper()})")
    print(f"{'='*60}\n")

    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        print(f"토픽 {topic_idx + 1}:")
        print(f"  {', '.join(top_words)}\n")

In [6]:
document_df = preprocessing.get_default_data()

📂 51개 파일 발견
🔄 텍스트 전처리 시작...
   옵션: HTML제거=True, URL제거=True, 숫자제거=True
   옵션: 불용어제거=True, Lemmatization=True, Stemming=False
✅ 전처리 완료:
   - 원본 문서 수: 51
   - 제거된 빈 문서: 0
   - 최종 문서 수: 51
   - 평균 단어 수: 1266.9


In [7]:
document_df.head()

,filename,opinion_text,processed_text,word_count
0,accuracy_garmin_nuvi_255W_gps,", and is very, very accurate .\n but for the m...",accurate part find garmin software provides ac...,543
1,bathroom_bestwestern_hotel_sfo,"The room was not overly big, but clean and ve...",room overly big clean comfortable bed great sh...,739
2,battery-life_amazon_kindle,After I plugged it in to my USB hub on my com...,plugged usb hub computer charge battery chargi...,876
3,battery-life_ipod_nano_8gb,short battery life I moved up from an 8gb .\...,short battery life moved gb love ipod except b...,620
4,battery-life_netbook_1005ha,"6GHz 533FSB cpu, glossy display, 3, Cell 23Wh ...",ghz fsb cpu glossy display cell wh li ion batt...,3243


In [8]:
# ==========================================
# 실행 
# ==========================================
lemmatizer, stop_words = LDATopicModeling(n_topics=5, random_state=42)

print("=" * 60)
print("LDA 토픽 모델링 시작")
print("=" * 60)


LDA 토픽 모델링 시작


In [9]:
# ==========================================
# 방법 2: Scikit-learn LDA
# ==========================================
print("\n[방법 2] Scikit-learn을 이용한 LDA\n")

sklearn_model, vectorizer, dtm = train_sklearn_lda(document_df["processed_text"])
doc_topics_sklearn = get_document_topics_sklearn(sklearn_model, dtm)

feature_names = vectorizer.get_feature_names_out()
display_topics(sklearn_model, feature_names, model_type="sklearn", n_top_words=5)

print("\n문서-토픽 분포 (첫 10개 문서):")
print(doc_topics_sklearn[:10])


[방법 2] Scikit-learn을 이용한 LDA

Scikit-learn LDA Perplexity: 694.5865

주요 토픽 및 키워드 (SKLEARN)

토픽 1:
  room, location, hotel, staff, service

토픽 2:
  video, price, free, sound, camera

토픽 3:
  battery, screen, life, button, keyboard

토픽 4:
  free, location, battery, life, room

토픽 5:
  mileage, interior, car, gas, comfortable


문서-토픽 분포 (첫 10개 문서):
[[5.05240546e-04 4.99567374e-04 9.97994766e-01 4.92903138e-04
  5.07523236e-04]
 [9.98624297e-01 3.45564556e-04 3.46546501e-04 3.39224782e-04
  3.44367034e-04]
 [3.11925716e-04 3.19546296e-04 9.98747620e-01 3.08805283e-04
  3.12102993e-04]
 [4.47842150e-04 5.10142094e-01 4.88517357e-01 4.43646636e-04
  4.49060340e-04]
 [8.38648658e-05 8.42757481e-05 9.99666725e-01 8.20794019e-05
  8.30548139e-05]
 [1.74488169e-04 1.74563217e-04 9.99303941e-01 1.71928624e-04
  1.75078593e-04]
 [1.88077551e-04 1.87972465e-04 1.87386638e-02 1.84946948e-04
  9.80700339e-01]
 [2.73953163e-04 2.73349557e-04 1.67472431e-02 2.68932782e-04
  9.82436521e-01]
 [2.4128982

In [23]:
def perform_advanced_pycaret_clustering(
    doc_topic_dist,
    documents=None,
    lda_model=None,
    feature_names=None,
    n_top_words=10
):
    """
    고급 PyCaret 클러스터링
    - 알고리즘별 성능 비교
    - 알고리즘별 / 클러스터별 대표 토픽 & 키워드 출력
    (display_topics 로직 참고)
    """

    import numpy as np
    import pandas as pd
    from pycaret.clustering import setup, create_model, assign_model
    from sklearn.metrics import silhouette_score, davies_bouldin_score

    print("\n" + "=" * 60)
    print("고급 클러스터링 분석 - 여러 알고리즘 비교")
    print("=" * 60 + "\n")

    # -------------------------
    # DataFrame 생성
    # -------------------------
    topic_columns = [f"Topic_{i+1}" for i in range(doc_topic_dist.shape[1])]
    df = pd.DataFrame(doc_topic_dist, columns=topic_columns)
    df["Document_ID"] = range(len(df))

    # -------------------------
    # PyCaret setup
    # -------------------------
    setup(
        data=df,
        session_id=42,
        normalize=True,
        transformation=False,
        ignore_features=["Document_ID"],
        verbose=False,
        html=False,
    )

    algorithms = ["kmeans", "ap", "meanshift", "sc", "hclust", "dbscan"]
    results = {}

    print("다양한 클러스터링 알고리즘 테스트 중...\n")

    for algo in algorithms:
        try:
            print("\n" + "=" * 60)
            print(f"{algo.upper()} 결과")
            print("=" * 60)

            if algo in ["kmeans", "sc", "hclust"]:
                model = create_model(algo, num_clusters=3)
            else:
                model = create_model(algo)

            predictions = assign_model(model)
            cluster_labels = predictions["Cluster"].values

            # -------------------------
            # 성능 평가
            # -------------------------
            if len(np.unique(cluster_labels)) == 1:
                silhouette = -1
                davies_bouldin = 999
            else:
                silhouette = silhouette_score(doc_topic_dist, cluster_labels)
                davies_bouldin = davies_bouldin_score(doc_topic_dist, cluster_labels)

            n_clusters = len(np.unique(cluster_labels))

            print(f"Clusters: {n_clusters}")
            print(f"Silhouette Score: {silhouette:.4f}")
            print(f"Davies-Bouldin Score: {davies_bouldin:.4f}")

            # -------------------------
            # 📌 클러스터별 대표 토픽 & 키워드
            # -------------------------
            print("\n📌 클러스터별 주요 토픽 및 키워드")

            for c in np.unique(cluster_labels):
                cluster_docs = doc_topic_dist[cluster_labels == c]
                mean_topic_dist = cluster_docs.mean(axis=0)

                top_topic_idx = mean_topic_dist.argmax()
                top_topic_no = top_topic_idx + 1

                print(f"\n- 클러스터 Cluster {c}")
                print(f"  ▶ 대표 토픽: Topic {top_topic_no}")

                # display_topics 방식 적용
                if lda_model is not None and feature_names is not None:
                    topic = lda_model.components_[top_topic_idx]
                    top_indices = topic.argsort()[-n_top_words:][::-1]
                    top_words = [feature_names[i] for i in top_indices]

                    print(f"  ▶ 키워드:")
                    print(f"    {', '.join(top_words)}")
                else:
                    print("  ▶ 키워드: (lda_model 또는 feature_names 없음)")

            results[algo] = {
                "model": model,
                "predictions": predictions,
                "silhouette_score": silhouette,
                "davies_bouldin_score": davies_bouldin,
                "n_clusters": n_clusters,
            }

        except Exception as e:
            print(f"\n✗ {algo} 실패: {str(e)[:80]}")
            continue

    # -------------------------
    # 알고리즘 성능 요약
    # -------------------------
    print("\n" + "=" * 60)
    print("알고리즘별 성능 비교 요약")
    print("=" * 60 + "\n")

    summary_df = pd.DataFrame([
        {
            "Algorithm": k.upper(),
            "N_Clusters": v["n_clusters"],
            "Silhouette": v["silhouette_score"],
            "Davies_Bouldin": v["davies_bouldin_score"],
        }
        for k, v in results.items()
    ]).sort_values("Silhouette", ascending=False)

    print(summary_df.to_string(index=False))

    best_algo = summary_df.iloc[0]["Algorithm"].lower()
    best_model_result = results[best_algo]

    print(f"\n최고 성능 알고리즘: {best_algo.upper()}")
    print(f"Silhouette Score: {best_model_result['silhouette_score']:.4f}")
    print(f"클러스터 개수: {best_model_result['n_clusters']}")

    return results, best_algo, best_model_result


In [ ]:
results, best_algo, best_model_result = perform_advanced_pycaret_clustering(
    doc_topic_dist=doc_topics_sklearn,
    documents=document_df["opinion_text"].tolist(),
    lda_model=sklearn_model,
    feature_names=vectorizer.get_feature_names_out(),
    n_top_words=10
)



고급 클러스터링 분석 - 여러 알고리즘 비교

다양한 클러스터링 알고리즘 테스트 중...


KMEANS 결과


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0       0.526            33.9661          0.9957            0           0   

   Completeness  
0             0  


Clusters: 3
Silhouette Score: 0.7602
Davies-Bouldin Score: 0.5954

📌 클러스터별 주요 토픽 및 키워드

- 클러스터 Cluster Cluster 0
  ▶ 대표 토픽: Topic 5
  ▶ 키워드:
    mileage, interior, car, gas, comfortable, seat, transmission, quality, comfort, parking

- 클러스터 Cluster Cluster 1
  ▶ 대표 토픽: Topic 1
  ▶ 키워드:
    room, location, hotel, staff, service, clean, friendly, food, helpful, small

- 클러스터 Cluster Cluster 2
  ▶ 대표 토픽: Topic 3
  ▶ 키워드:
    battery, screen, life, button, keyboard, size, speed, direction, feature, voice

AP 결과


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0      0.6196            63.2329          0.4684            0           0   

   Completeness  
0             0  


Clusters: 5
Silhouette Score: 0.9017
Davies-Bouldin Score: 0.2581

📌 클러스터별 주요 토픽 및 키워드

- 클러스터 Cluster Cluster 0
  ▶ 대표 토픽: Topic 2
  ▶ 키워드:
    video, price, free, sound, camera, quality, ipod, satellite, wine, feature

- 클러스터 Cluster Cluster 1
  ▶ 대표 토픽: Topic 5
  ▶ 키워드:
    mileage, interior, car, gas, comfortable, seat, transmission, quality, comfort, parking

- 클러스터 Cluster Cluster 2
  ▶ 대표 토픽: Topic 2
  ▶ 키워드:
    video, price, free, sound, camera, quality, ipod, satellite, wine, feature

- 클러스터 Cluster Cluster 3
  ▶ 대표 토픽: Topic 3
  ▶ 키워드:
    battery, screen, life, button, keyboard, size, speed, direction, feature, voice

- 클러스터 Cluster Cluster 4
  ▶ 대표 토픽: Topic 1
  ▶ 키워드:
    room, location, hotel, staff, service, clean, friendly, food, helpful, small

MEANSHIFT 결과


...

Clusters: 3
Silhouette Score: 0.7707
Davies-Bouldin Score: 0.4131

📌 클러스터별 주요 토픽 및 키워드

- 클러스터 Cluster Cluster 0
  ▶ 대표 토픽: Topic 1
  ▶ 키워드:
    room, location, hotel, staff, service, clean, friendly, food, helpful, small

- 클러스터 Cluster Cluster 1
  ▶ 대표 토픽: Topic 3
  ▶ 키워드:
    battery, screen, life, button, keyboard, size, speed, direction, feature, voice

- 클러스터 Cluster Cluster 2
  ▶ 대표 토픽: Topic 5
  ▶ 키워드:
    mileage, interior, car, gas, comfortable, seat, transmission, quality, comfort, parking

HCLUST 결과


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0      0.5367            35.4875          0.7622            0           0   

   Completeness  
0             0  


Clusters: 3
Silhouette Score: 0.7707
Davies-Bouldin Score: 0.4131

📌 클러스터별 주요 토픽 및 키워드

- 클러스터 Cluster Cluster 0
  ▶ 대표 토픽: Topic 3
  ▶ 키워드:
    battery, screen, life, button, keyboard, size, speed, direction, feature, voice

- 클러스터 Cluster Cluster 1
  ▶ 대표 토픽: Topic 1
  ▶ 키워드:
    room, location, hotel, staff, service, clean, friendly, food, helpful, small

- 클러스터 Cluster Cluster 2
  ▶ 대표 토픽: Topic 5
  ▶ 키워드:
    mileage, interior, car, gas, comfortable, seat, transmission, quality, comfort, parking

DBSCAN 결과


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0      0.4249            20.6256          1.0917            0           0   

   Completeness  
0             0  


Clusters: 5
Silhouette Score: 0.1669
Davies-Bouldin Score: 1.4138

📌 클러스터별 주요 토픽 및 키워드

- 클러스터 Cluster Cluster -1
  ▶ 대표 토픽: Topic 2
  ▶ 키워드:
    video, price, free, sound, camera, quality, ipod, satellite, wine, feature

- 클러스터 Cluster Cluster 0
  ▶ 대표 토픽: Topic 3
  ▶ 키워드:
    battery, screen, life, button, keyboard, size, speed, direction, feature, voice

- 클러스터 Cluster Cluster 1
  ▶ 대표 토픽: Topic 3
  ▶ 키워드:
    battery, screen, life, button, keyboard, size, speed, direction, feature, voice

- 클러스터 Cluster Cluster 2
  ▶ 대표 토픽: Topic 5
  ▶ 키워드:
    mileage, interior, car, gas, comfortable, seat, transmission, quality, comfort, parking

- 클러스터 Cluster Cluster 3
  ▶ 대표 토픽: Topic 1
  ▶ 키워드:
    room, location, hotel, staff, service, clean, friendly, food, helpful, small

알고리즘별 성능 비교 요약

Algorithm  N_Clusters  Silhouette  Davies_Bouldin
       AP           5    0.901711        0.258067
       SC           3    0.770687        0.413121
   HCLUST           3    0.770687        0.413121
   KM

In [38]:
def perform_advanced_pycaret_clustering(
    doc_topic_dist,
    documents=None,
    file_names=None,          # [추가]
    lda_model=None,
    feature_names=None,
    n_top_words=10,
    save_dir="../results"     # [추가]
):
    """
    고급 PyCaret 클러스터링
    - 출력 로직 유지
    - 결과 CSV 저장만 추가
    """

    import os
    import numpy as np
    import pandas as pd
    from pycaret.clustering import setup, create_model, assign_model
    from sklearn.metrics import silhouette_score, davies_bouldin_score

    os.makedirs(save_dir, exist_ok=True)   # [추가]

    print("\n" + "=" * 60)
    print("고급 클러스터링 분석 - 여러 알고리즘 비교")
    print("=" * 60 + "\n")

    # -------------------------
    # DataFrame 생성
    # -------------------------
    topic_columns = [f"Topic_{i+1}" for i in range(doc_topic_dist.shape[1])]
    df = pd.DataFrame(doc_topic_dist, columns=topic_columns)
    df["Document_ID"] = range(len(df))

    setup(
        data=df,
        session_id=42,
        normalize=True,
        transformation=False,
        ignore_features=["Document_ID"],
        verbose=False,
        html=False,
    )

    algorithms = ["kmeans", "ap", "meanshift", "sc", "hclust", "dbscan"]
    results = {}

    for algo in algorithms:
        try:
            print("\n" + "=" * 60)
            print(f"{algo.upper()} 결과")
            print("=" * 60)

            if algo in ["kmeans", "sc", "hclust"]:
                model = create_model(algo, num_clusters=3)
            else:
                model = create_model(algo)

            predictions = assign_model(model)

            # 🔴 중요: Cluster 문자열 → 숫자 정규화
            cluster_labels = (
                predictions["Cluster"]
                .astype(str)
                .str.replace("Cluster", "", regex=False)
                .astype(int)
                .values
            )

            # -------------------------
            # 성능 평가 (기존 그대로)
            # -------------------------
            if len(np.unique(cluster_labels)) == 1:
                silhouette = -1
                davies_bouldin = 999
            else:
                silhouette = silhouette_score(doc_topic_dist, cluster_labels)
                davies_bouldin = davies_bouldin_score(doc_topic_dist, cluster_labels)

            n_clusters = len(np.unique(cluster_labels))

            print(f"Clusters: {n_clusters}")
            print(f"Silhouette Score: {silhouette:.4f}")
            print(f"Davies-Bouldin Score: {davies_bouldin:.4f}")

            # -------------------------
            # 📌 클러스터별 주요 토픽 & 키워드 (기존 그대로)
            # -------------------------
            print("\n📌 클러스터별 주요 토픽 및 키워드")

            for c in np.unique(cluster_labels):
                cluster_docs = doc_topic_dist[cluster_labels == c]
                mean_topic_dist = cluster_docs.mean(axis=0)
                top_topic_idx = mean_topic_dist.argmax()
                top_topic_no = top_topic_idx + 1

                print(f"\n- 클러스터 Cluster {c}")
                print(f"  ▶ 대표 토픽: Topic {top_topic_no}")

                if lda_model is not None and feature_names is not None:
                    topic = lda_model.components_[top_topic_idx]
                    top_indices = topic.argsort()[-n_top_words:][::-1]
                    top_words = [feature_names[i] for i in top_indices]
                    print(f"  ▶ 키워드:")
                    print(f"    {', '.join(top_words)}")
                else:
                    print("  ▶ 키워드: (lda_model 또는 feature_names 없음)")

            # =====================================================
            # ✅ [추가] 문서–클러스터 매칭 CSV 저장
            # =====================================================
            # =====================================================
            # 1️⃣ 클러스터별 대표 토픽 / 키워드 계산
            # =====================================================
            cluster_topic_map = {}

            for c in np.unique(cluster_labels):
                cluster_docs = doc_topic_dist[cluster_labels == c]
                mean_topic_dist = cluster_docs.mean(axis=0)

                top_topic_idx = mean_topic_dist.argmax()
                top_topic_no = top_topic_idx + 1

                keywords = ""
                if lda_model is not None and feature_names is not None:
                    topic = lda_model.components_[top_topic_idx]
                    top_indices = topic.argsort()[-n_top_words:][::-1]
                    keywords = ", ".join(feature_names[i] for i in top_indices)

                cluster_topic_map[int(c)] = {
                    "Top_Topic": top_topic_no,
                    "Top_Topic_Score": float(mean_topic_dist[top_topic_idx]),  # ✅ 이 줄
                    "Topic_Keywords": keywords
                }

            # =====================================================
            # 2️⃣ 문서 단위 DataFrame 생성
            # =====================================================
            save_df = pd.DataFrame({
                "Cluster": cluster_labels
            })

            if documents is not None:
                save_df["Document_Text"] = documents

            if file_names is not None:
                save_df["File_Name"] = file_names

            # =====================================================
            # 3️⃣ 각 문서에 대표 토픽 / 키워드 매핑
            # =====================================================
            save_df["Top_Topic"] = save_df["Cluster"].map(
                lambda c: cluster_topic_map[int(c)]["Top_Topic"]
            )

            save_df["Top_Topic_Score"] = save_df["Cluster"].map(
                lambda c: cluster_topic_map[int(c)]["Top_Topic_Score"]
            )

            save_df["Topic_Keywords"] = save_df["Cluster"].map(
                lambda c: cluster_topic_map[int(c)]["Topic_Keywords"]
            )

            # 컬럼 순서 정리
            save_df = save_df[
                [c for c in ["File_Name","Document_Text","Cluster","Top_Topic","Top_Topic_Score","Topic_Keywords"
                ] if c in save_df.columns]
            ]

            save_df = save_df.sort_values(
                by=["Cluster", "File_Name"]
            ).reset_index(drop=True)

            # =====================================================
            # 4️⃣ CSV 저장
            # =====================================================
            from datetime import datetime

            date_str = datetime.now().strftime("%Y%m%d")

            save_df.to_csv(
                f"{save_dir}/{algo}_document_cluster_mapping_{date_str}.csv",
                index=False,
                encoding="utf-8-sig"
            )



            results[algo] = {
                "model": model,
                "predictions": predictions,
                "silhouette_score": silhouette,
                "davies_bouldin_score": davies_bouldin,
                "n_clusters": n_clusters,
            }

        except Exception as e:
            print(f"\n✗ {algo} 실패: {str(e)[:80]}")
            continue

    # -------------------------
    # 알고리즘 성능 요약 (기존 그대로)
    # -------------------------
    print("\n" + "=" * 60)
    print("알고리즘별 성능 비교 요약")
    print("=" * 60 + "\n")

    summary_df = pd.DataFrame([
        {
            "Algorithm": k.upper(),
            "N_Clusters": v["n_clusters"],
            "Silhouette": v["silhouette_score"],
            "Davies_Bouldin": v["davies_bouldin_score"],
        }
        for k, v in results.items()
    ]).sort_values("Silhouette", ascending=False)

    print(summary_df.to_string(index=False))

    best_algo = summary_df.iloc[0]["Algorithm"].lower()
    best_model_result = results[best_algo]

    print(f"\n최고 성능 알고리즘: {best_algo.upper()}")
    print(f"Silhouette Score: {best_model_result['silhouette_score']:.4f}")
    print(f"클러스터 개수: {best_model_result['n_clusters']}")

    return results, best_algo, best_model_result


In [39]:
results, best_algo, best_model_result = perform_advanced_pycaret_clustering(
    doc_topic_dist=doc_topics_sklearn,
    documents=document_df["opinion_text"].tolist(),
    file_names=document_df["filename"].tolist(),  # ← 이 컬럼명 맞게
    lda_model=sklearn_model,
    feature_names=vectorizer.get_feature_names_out(),
    n_top_words=10,
    save_dir="../results"
)



고급 클러스터링 분석 - 여러 알고리즘 비교


KMEANS 결과


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0       0.526            33.9661          0.9957            0           0   

   Completeness  
0             0  


Clusters: 3
Silhouette Score: 0.7602
Davies-Bouldin Score: 0.5954

📌 클러스터별 주요 토픽 및 키워드

- 클러스터 Cluster 0
  ▶ 대표 토픽: Topic 5
  ▶ 키워드:
    mileage, interior, car, gas, comfortable, seat, transmission, quality, comfort, parking

- 클러스터 Cluster 1
  ▶ 대표 토픽: Topic 1
  ▶ 키워드:
    room, location, hotel, staff, service, clean, friendly, food, helpful, small

- 클러스터 Cluster 2
  ▶ 대표 토픽: Topic 3
  ▶ 키워드:
    battery, screen, life, button, keyboard, size, speed, direction, feature, voice

AP 결과


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0      0.6196            63.2329          0.4684            0           0   

   Completeness  
0             0  


Clusters: 5
Silhouette Score: 0.9017
Davies-Bouldin Score: 0.2581

📌 클러스터별 주요 토픽 및 키워드

- 클러스터 Cluster 0
  ▶ 대표 토픽: Topic 2
  ▶ 키워드:
    video, price, free, sound, camera, quality, ipod, satellite, wine, feature

- 클러스터 Cluster 1
  ▶ 대표 토픽: Topic 5
  ▶ 키워드:
    mileage, interior, car, gas, comfortable, seat, transmission, quality, comfort, parking

- 클러스터 Cluster 2
  ▶ 대표 토픽: Topic 2
  ▶ 키워드:
    video, price, free, sound, camera, quality, ipod, satellite, wine, feature

- 클러스터 Cluster 3
  ▶ 대표 토픽: Topic 3
  ▶ 키워드:
    battery, screen, life, button, keyboard, size, speed, direction, feature, voice

- 클러스터 Cluster 4
  ▶ 대표 토픽: Topic 1
  ▶ 키워드:
    room, location, hotel, staff, service, clean, friendly, food, helpful, small

MEANSHIFT 결과


...

Clusters: 3
Silhouette Score: 0.7707
Davies-Bouldin Score: 0.4131

📌 클러스터별 주요 토픽 및 키워드

- 클러스터 Cluster 0
  ▶ 대표 토픽: Topic 1
  ▶ 키워드:
    room, location, hotel, staff, service, clean, friendly, food, helpful, small

- 클러스터 Cluster 1
  ▶ 대표 토픽: Topic 3
  ▶ 키워드:
    battery, screen, life, button, keyboard, size, speed, direction, feature, voice

- 클러스터 Cluster 2
  ▶ 대표 토픽: Topic 5
  ▶ 키워드:
    mileage, interior, car, gas, comfortable, seat, transmission, quality, comfort, parking

HCLUST 결과


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0      0.5367            35.4875          0.7622            0           0   

   Completeness  
0             0  


Clusters: 3
Silhouette Score: 0.7707
Davies-Bouldin Score: 0.4131

📌 클러스터별 주요 토픽 및 키워드

- 클러스터 Cluster 0
  ▶ 대표 토픽: Topic 3
  ▶ 키워드:
    battery, screen, life, button, keyboard, size, speed, direction, feature, voice

- 클러스터 Cluster 1
  ▶ 대표 토픽: Topic 1
  ▶ 키워드:
    room, location, hotel, staff, service, clean, friendly, food, helpful, small

- 클러스터 Cluster 2
  ▶ 대표 토픽: Topic 5
  ▶ 키워드:
    mileage, interior, car, gas, comfortable, seat, transmission, quality, comfort, parking

DBSCAN 결과


   Silhouette  Calinski-Harabasz  Davies-Bouldin  Homogeneity  Rand Index  \
0      0.4249            20.6256          1.0917            0           0   

   Completeness  
0             0  


Clusters: 5
Silhouette Score: 0.1669
Davies-Bouldin Score: 1.4138

📌 클러스터별 주요 토픽 및 키워드

- 클러스터 Cluster -1
  ▶ 대표 토픽: Topic 2
  ▶ 키워드:
    video, price, free, sound, camera, quality, ipod, satellite, wine, feature

- 클러스터 Cluster 0
  ▶ 대표 토픽: Topic 3
  ▶ 키워드:
    battery, screen, life, button, keyboard, size, speed, direction, feature, voice

- 클러스터 Cluster 1
  ▶ 대표 토픽: Topic 3
  ▶ 키워드:
    battery, screen, life, button, keyboard, size, speed, direction, feature, voice

- 클러스터 Cluster 2
  ▶ 대표 토픽: Topic 5
  ▶ 키워드:
    mileage, interior, car, gas, comfortable, seat, transmission, quality, comfort, parking

- 클러스터 Cluster 3
  ▶ 대표 토픽: Topic 1
  ▶ 키워드:
    room, location, hotel, staff, service, clean, friendly, food, helpful, small

알고리즘별 성능 비교 요약

Algorithm  N_Clusters  Silhouette  Davies_Bouldin
       AP           5    0.901711        0.258067
       SC           3    0.770687        0.413121
   HCLUST           3    0.770687        0.413121
   KMEANS           3    0.760173        0.59

In [31]:
print(results)

{'kmeans': {'model': KMeans(n_clusters=3, random_state=42), 'predictions':      Topic_1   Topic_2   Topic_3   Topic_4   Topic_5    Cluster
0   0.000505  0.000500  0.997995  0.000493  0.000508  Cluster 2
1   0.998624  0.000346  0.000347  0.000339  0.000344  Cluster 1
2   0.000312  0.000320  0.998748  0.000309  0.000312  Cluster 2
3   0.000448  0.510142  0.488517  0.000444  0.000449  Cluster 2
4   0.000084  0.000084  0.999667  0.000082  0.000083  Cluster 2
5   0.000174  0.000175  0.999304  0.000172  0.000175  Cluster 2
6   0.000188  0.000188  0.018739  0.000185  0.980700  Cluster 0
7   0.000274  0.000273  0.016747  0.000269  0.982437  Cluster 0
8   0.024129  0.000274  0.975050  0.000269  0.000278  Cluster 2
9   0.000525  0.000521  0.997917  0.000514  0.000522  Cluster 2
10  0.000363  0.000365  0.998551  0.000359  0.000363  Cluster 2
11  0.000513  0.000517  0.997950  0.000505  0.000514  Cluster 2
12  0.000377  0.000375  0.998505  0.000370  0.000373  Cluster 2
13  0.999111  0.000223  0.000

✅ 1. 첫 번째 코드 그룹 — LDA 학습 + 토픽 출력 기능

이 그룹에는 다음 함수들이 포함됨:

✔️ train_sklearn_lda()

텍스트 데이터를 CountVectorizer로 DTM(document-term matrix) 만들고

LDA 토픽 모델을 학습하고

Perplexity 출력하고

(LDA model, vectorizer, dtm)을 반환함
→ 즉, 비지도 학습 (Unsupervised Learning)

✔️ get_document_topics_sklearn()

LDA 모델로부터 문서별 토픽 분포 행렬 생성
→ shape = (n_documents, n_topics)
→ 이것도 비지도 학습 과정

✔️ display_topics()

LDA 모델에서 각 토픽의 주요 단어 top N 출력
→ 사람이 토픽 해석하는 용도
→ 역시 비지도 학습 분석

📌 정리하면?

첫 번째 코드 블록은 LDA 자체를 학습하고 토픽을 해석하는 역할이다.

✅ 2. 두 번째 코드 — 지도학습(분류) 예시
✔️ supervised_learning_example()

이 함수는 역할이 완전히 다름.

LDA가 만들어 준 문서-토픽 벡터(doc_topic_dist)를 feature로 사용

labels(감성/카테고리 같은 supervised label)과 함께

Train/Test split 수행

Logistic Regression 훈련

Accuracy, Classification Report 출력

→ 완전 다른 종류의 ML 작업: 지도 학습(Supervised Learning)
→ 목표: “LDA가 생성한 latent topic embedding이 분류 문제에서 얼마나 유용한가 평가”

🎯 딱 핵심 차이
기능	첫 번째 코드	두 번째 코드
학습 종류	비지도 학습(Unsupervised LDA)	지도 학습(Supervised Classification)
입력	문서 텍스트	LDA 출력된 토픽 분포 + 레이블
출력	토픽 단어, Perplexity, 토픽-문서 분포	Accuracy, Classification Report
목적	토픽 모델링	분류 예측
사용 모델	LDA	Logistic Regression